In [2]:
import pandas as pd
import re   
import pymorphy3
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib


# лемматизатор
morph = pymorphy3.MorphAnalyzer()

df = pd.read_csv("jkh_hierarchical_dataset.csv", encoding="utf-8-sig")
print(f"Строк датасета: {len(df)}")

def lemmatize_text(text):           
    text = text.lower()
    text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text)
    words = text.split()
    
    lemmatized_words = []
    for word in words:
        parsed = morph.parse(word)[0] # Берем первый (самый вероятный) разбор слова
        lemmatized_words.append(parsed.normal_form)
            
    return " ".join(lemmatized_words)

df['cleaned_text'] = df['text'].apply(lemmatize_text)

sub_to_main_mapping = df.set_index('sub_category')['main_category'].to_dict()

# Train/Test
train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['sub_category'])
train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=42, stratify=train_val_df['sub_category'])

# 2. обучение модели
main_model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 3), analyzer='word', min_df=1), 
    LogisticRegression(class_weight='balanced', C=10, max_iter=100)
)
main_model.fit(train_df['cleaned_text'], train_df['main_category'])

# 3. обучение модели подкатегорий
global_sub_model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 3), analyzer='word', min_df=1), 
    LogisticRegression(class_weight='balanced', C=10, max_iter=100)
)
global_sub_model.fit(train_df['cleaned_text'], train_df['sub_category'])

# 4. коррекция
def classify_request_with_correction(text, sub_confidence_threshold=0.75):
    cleaned = lemmatize_text(text)
    
    # Шаг A: Предсказание категории
    main_pred = main_model.predict([cleaned])[0]
    main_probs = main_model.predict_proba([cleaned])
    main_class_idx = list(main_model.classes_).index(main_pred)
    main_conf = main_probs[0][main_class_idx]
    
    # Шаг B: Предсказание подкатегории
    global_sub_pred = global_sub_model.predict([cleaned])[0]
    global_sub_probs = global_sub_model.predict_proba([cleaned])
    global_sub_idx = list(global_sub_model.classes_).index(global_sub_pred)
    global_sub_conf = global_sub_probs[0][global_sub_idx]
    
    # логика коррекции
    if global_sub_conf > sub_confidence_threshold:
        correct_main_cat = sub_to_main_mapping[global_sub_pred]
        if main_pred != correct_main_cat:
            main_pred = correct_main_cat
            main_conf = global_sub_conf  
            
    return main_pred, main_conf, global_sub_pred, global_sub_conf

# 5. расчёт точности
def get_metrics(dataset):
    preds_main = []
    preds_sub = []
    for _, row in dataset.iterrows():
        m_p, _, s_p, _ = classify_request_with_correction(row['text'])
        preds_main.append(m_p)
        preds_sub.append(s_p)
    return accuracy_score(dataset['main_category'], preds_main), accuracy_score(dataset['sub_category'], preds_sub)

train_main_acc, train_sub_acc = get_metrics(train_df)
val_main_acc, val_sub_acc = get_metrics(val_df)
test_main_acc, test_sub_acc = get_metrics(test_df)

print(f"\n✅ РЕЗУЛЬТАТЫ НА ОБУЧАЮЩЕЙ ВЫБОРКЕ:")
print(f"-> Точность (Главные категории): {train_main_acc:.1%}")
print(f"-> Точность (Подкатегории): {train_sub_acc:.1%}")

print(f"\n✅ РЕЗУЛЬТАТЫ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ:")
print(f"-> Точность (Главные категории): {val_main_acc:.1%}")
print(f"-> Точность (Подкатегории): {val_sub_acc:.1%}")

print(f"\n✅ РЕЗУЛЬТАТЫ СИСТЕМЫ С КОРРЕКЦИЕЙ:")
print(f"-> Точность (Главные категории): {test_main_acc:.1%}")
print(f"-> Точность (Подкатегории): {test_sub_acc:.1%}")

# 7. тест (можно любые примеры вписать)
print("\n=== ПРОВЕРКА СЛОЖНЫХ ОБРАЩЕНИЙ ===")
demo_questions = [
    "Не пришла бумажная квитанция за прошлый месяц, сделайте дубликат счета",
    "Во дворе шлагбаум заклинило, машины не могут выехать со стоянки",
    "На 4 этаже перегорели лампочки, в подъезде кромешная тьма"
]

for question in demo_questions:
    m_cat, m_conf, s_cat, s_conf = classify_request_with_correction(question)
    print(f"💬 Текст жильца: '{question}'")
    print(f"📂 Корневой класс: [{m_cat}]")
    print(f"🎯 Точная подкатегория: -> {s_cat} (Уверенность: {s_conf:.1%})")
    print("-" * 70)


joblib.dump({"main": main_model, "sub": global_sub_model, "map": sub_to_main_mapping}, "model.pkl")

Строк датасета: 2380

✅ РЕЗУЛЬТАТЫ НА ОБУЧАЮЩЕЙ ВЫБОРКЕ:
-> Точность (Главные категории): 97.1%
-> Точность (Подкатегории): 99.1%

✅ РЕЗУЛЬТАТЫ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ:
-> Точность (Главные категории): 95.3%
-> Точность (Подкатегории): 96.9%

✅ РЕЗУЛЬТАТЫ СИСТЕМЫ С КОРРЕКЦИЕЙ:
-> Точность (Главные категории): 93.9%
-> Точность (Подкатегории): 97.7%

=== ПРОВЕРКА СЛОЖНЫХ ОБРАЩЕНИЙ ===
💬 Текст жильца: 'Не пришла бумажная квитанция за прошлый месяц, сделайте дубликат счета'
📂 Корневой класс: [Квитанции, начисления]
🎯 Точная подкатегория: -> Запрос квитанции  (Уверенность: 76.0%)
----------------------------------------------------------------------
💬 Текст жильца: 'Во дворе шлагбаум заклинило, машины не могут выехать со стоянки'
📂 Корневой класс: [Домофоны, шлагбаумы, ворота, видеонаблюдение]
🎯 Точная подкатегория: -> Неисправность шлагбаумов (Уверенность: 17.2%)
----------------------------------------------------------------------
💬 Текст жильца: 'На 4 этаже перегорели лампочки, в подъ

['model.pkl']

Вызов модели

In [13]:
import re
import joblib
import pymorphy3

# 1. Загрузка компонентов модели
data = joblib.load("model.pkl")
main_model, global_sub_model, sub_to_main_mapping = data["main"], data["sub"], data["map"]

# 2. Инициализация лемматизатора
morph = pymorphy3.MorphAnalyzer()

def classify_request(text, sub_confidence_threshold=0.75):
    # Очистка и лемматизация текста
    cleaned = " ".join([morph.parse(w)[0].normal_form for w in re.sub(r'[^а-яёa-z0-9\s]', ' ', text.lower()).split()])
    
    # Предсказание главной категории (берём, чтобы получить строку, а не список)
    main_pred = main_model.predict([cleaned])[0]
    main_probs = main_model.predict_proba([cleaned])
    main_conf = main_probs[0][list(main_model.classes_).index(main_pred)]
    
    # Предсказание подкатегории (берём, чтобы получить строку, а не список)
    sub_pred = global_sub_model.predict([cleaned])[0]
    sub_probs = global_sub_model.predict_proba([cleaned])
    sub_conf = sub_probs[0][list(global_sub_model.classes_).index(sub_pred)]
    
    # Логика коррекции на основе подкатегории
    if sub_conf > sub_confidence_threshold:
        correct_main_cat = sub_to_main_mapping.get(sub_pred, main_pred)
        if main_pred != correct_main_cat:
            main_pred, main_conf = correct_main_cat, sub_conf
            
    return main_pred, main_conf, sub_pred, sub_conf

# === ПРИМЕР ИСПОЛЬЗОВАНИЯ ===
if __name__ == "__main__":
    text = "На 4 этаже перегорели лампочки, в подъезде кромешная тьма"
    m_cat, m_conf, s_cat, s_conf = classify_request(text)
    
    print(f"💬 Текст: '{text}'")
    print(f"📂 Главная категория: {m_cat} ({m_conf:.1%})")
    print(f"🎯 Подкатегория: {s_cat} ({s_conf:.1%})")


💬 Текст: 'На 4 этаже перегорели лампочки, в подъезде кромешная тьма'
📂 Главная категория: Освещение, электричество (55.4%)
🎯 Подкатегория: Неисправность освещения в подъезде / во дворе  (31.4%)


Генерация ответа

In [ ]:
import ollama 


def generate_validation_response(user_text, main_cat, sub_cat):
    # Промпт, который заставляет LLM переспросить
    system_prompt = f"""
    Ты — интеллектуальный ассистент службы поддержки. Твоя задача — вежливо уточнить у пользователя, правильно ли система классифицировала его проблему.
    
    Результат автоматической классификации:
    - Главная категория: {main_cat}
    - Подкатегория: {sub_cat}
    
    Инструкция:
    1. Спроси, используй понятный человеку язык, а не технические теги, начни с фразы: если я правильно понял у вас возникла проблема с ...).
    2. В конце ОБЯЗАТЕЛЬНО задай прямой вопрос: правильно ли распознана проблема, или требуется её скорректировать.
    3. Будь лаконичен. Не предлагай решение проблемы на этом этапе, только валидация.
    """

    # Запрос к локальной модели через Ollama
    response = ollama.chat(
        model="qwen2.5:7b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text},
        ],
        options={
            "temperature": 0.4
        },
    )

    # В Ollama ответ лежит в ['message']['content']
    return response["message"]["content"]




generate_validation_response(text, m_cat, s_cat)

'Если я правильно понял, у вас возникла проблема с неисправностью освещения в подъезде. Правильно ли распознана проблема?'